# ABRG corpus GAE pilot (visual)

Build behavioral graphs from Frida traces, train a Graph Autoencoder (GAE), and compare train vs held-out test reconstruction error.

**Active dataset:** reads `datasets/CURRENT` (v2: 168 sessions, hook_apis.js v3).

**Features (v0.2.1):** graph stores raw `w_cum` / `act_count`; GAE tensor uses **transition probabilities** (edges) and **activity fractions** (nodes).

```bash
cd /path/to/adaptive-behavioral-graph-analysis
source .venv/bin/activate
pip install -r abrg/requirements-notebook.txt
jupyter notebook notebooks/corpus_gae_pilot.ipynb
```

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Resolve repo root whether the kernel cwd is repo root or notebooks/
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "abrg").is_dir() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from abrg.autoencoder import build_gae, graph_reconstruction_error, train_gae_multi
from abrg.config import DEFAULT_WINDOW_SEC, GAE_EPOCHS, GAE_HIDDEN_DIM, GAE_LR
from abrg.corpus import build_corpus_graphs
from abrg.dataset_paths import current_dataset_version, current_sessions_dir
from abrg.features import graph_to_tensors, node_feature_dim
from abrg.registry import GRAPH_CATEGORY_UNIVERSE
from abrg.run_corpus_pilot import (
    SPLIT_SEED,
    TEST_RATIO,
    build_report,
    count_ln2_failures,
    distribution_stats,
    record_to_tensors,
    split_train_test_by_app,
)
from abrg.windows import WindowMode

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

print(f"Repo root: {REPO_ROOT}")
print(f"Dataset: {current_dataset_version()} → {current_sessions_dir()}")
print(f"Graph nodes: {len(GRAPH_CATEGORY_UNIVERSE)}")

## Configuration

Adjust these knobs, then re-run the cells below.

In [ ]:
SESSIONS_DIR = current_sessions_dir()  # datasets/CURRENT → e.g. datasets/v2/sessions
OUTPUT_DIR = REPO_ROOT / f"abrg/output/notebook_pilot_{current_dataset_version()}"

# Windowing: set WHOLE_SESSION=True for one graph per app
WHOLE_SESSION = False
WINDOW_SEC = DEFAULT_WINDOW_SEC  # 60s processing windows when WHOLE_SESSION=False
SNAPSHOTS = not WHOLE_SESSION

# Train / test
EPOCHS = GAE_EPOCHS
TEST_RATIO = TEST_RATIO
SEED = SPLIT_SEED

window_mode = WindowMode.WHOLE_SESSION if WHOLE_SESSION else WindowMode.TIME_SEC
mode_label = (
    f"whole session (1 graph/app)"
    if WHOLE_SESSION
    else f"multi-window snapshots ({WINDOW_SEC}s)"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(mode_label)
print(f"Sessions: {SESSIONS_DIR}")

## A. Build graphs

In [ ]:
t0 = time.time()
records = build_corpus_graphs(
    SESSIONS_DIR.resolve(),
    window_mode=window_mode,
    window_sec=WINDOW_SEC,
    snapshots=SNAPSHOTS,
)
build_time = time.time() - t0
build = build_report(
    records,
    snapshots=SNAPSHOTS,
    window_mode=window_mode.value,
    window_sec=WINDOW_SEC,
)
build["build_wall_sec"] = round(build_time, 2)

gae_eligible = [r for r in records if r.gae_eligible]
trainable = [r for r in records if r.trainable]

display(pd.DataFrame([build]).T.rename(columns={0: "value"}))

# Funnel chart
funnel = pd.Series(
    {
        "sessions": build["total_sessions"],
        "snapshots built": build["total_snapshots"],
        "trainable snapshots": build["trainable_snapshots"],
        "GAE-eligible": build["gae_eligible_snapshots"],
    }
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=funnel.index, y=funnel.values, ax=ax, hue=funnel.index, legend=False, palette="Blues_d")
ax.set_ylabel("count")
ax.set_title(f"Corpus funnel — {mode_label}")
for i, v in enumerate(funnel.values):
    ax.text(i, v + max(funnel.values) * 0.01, str(int(v)), ha="center", fontsize=10)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## B. Train / test split (by app)

In [ ]:
train_recs, test_recs, train_apps, test_apps = split_train_test_by_app(
    records, TEST_RATIO, SEED
)

split_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "apps": [len(train_apps), len(test_apps)],
        "snapshots": [len(train_recs), len(test_recs)],
    }
)
display(split_df)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=split_df, x="split", y="apps", ax=axes[0], hue="split", legend=False, palette=["#4c72b0", "#dd8452"])
axes[0].set_title("Apps per split")
sns.barplot(data=split_df, x="split", y="snapshots", ax=axes[1], hue="split", legend=False, palette=["#4c72b0", "#dd8452"])
axes[1].set_title("Snapshots per split")
plt.suptitle(f"80/20 app-level holdout (seed={SEED})")
plt.tight_layout()
plt.show()

## C. Train GAE

In [ ]:
train_tensors = [record_to_tensors(r) for r in train_recs]
train_pairs = [(x, ei) for x, ei, _ in train_tensors]

model = build_gae(node_feature_dim(), GAE_HIDDEN_DIM)
t1 = time.time()
curve, final_train_mean = train_gae_multi(model, train_pairs, EPOCHS, GAE_LR)
train_time = time.time() - t1
converged = len(curve) >= 2 and curve[-1] < curve[0] * 0.85

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(curve) + 1), curve, color="#4c72b0", lw=1.5)
ax.axhline(np.log(2), color="gray", ls="--", lw=1, label="ln(2) random guess")
ax.set_xlabel("epoch")
ax.set_ylabel("mean reconstruction loss")
ax.set_title(f"GAE training ({EPOCHS} epochs, {train_time:.1f}s)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Loss {curve[0]:.4f} → {curve[-1]:.4f} | converged={converged}")

## D. Evaluate — train vs test reconstruction error

In [ ]:
train_errors = [graph_reconstruction_error(model, x, ei) for x, ei, _ in train_tensors]
test_tensors = [record_to_tensors(r) for r in test_recs]
test_errors = [graph_reconstruction_error(model, x, ei) for x, ei, _ in test_tensors]

train_stats = distribution_stats(train_errors)
test_stats = distribution_stats(test_errors)

summary = pd.DataFrame(
    {
        "train": train_stats,
        "test": test_stats,
    }
).T[["count", "min", "median", "mean", "max"]]
display(summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overlaid histograms
err_df = pd.concat(
    [
        pd.DataFrame({"error": train_errors, "split": "train"}),
        pd.DataFrame({"error": test_errors, "split": "test"}),
    ],
    ignore_index=True,
)
sns.histplot(data=err_df, x="error", hue="split", kde=True, ax=axes[0], bins=30, alpha=0.5)
axes[0].axvline(np.log(2), color="gray", ls="--", label="ln(2)")
axes[0].set_title("Reconstruction error distribution")
axes[0].legend()

# Box plot
sns.boxplot(data=err_df, x="split", y="error", ax=axes[1], palette=["#4c72b0", "#dd8452"])
axes[1].axhline(np.log(2), color="gray", ls="--")
axes[1].set_title("Train vs test (median comparison)")
plt.tight_layout()
plt.show()

ratio = test_stats["median"] / train_stats["median"] if train_stats["median"] else float("nan")
print(
    f"Train median={train_stats['median']:.4f} | Test median={test_stats['median']:.4f} | "
    f"ratio={ratio:.3f} | ln2-fail train={count_ln2_failures(train_errors)}/{len(train_errors)} "
    f"test={count_ln2_failures(test_errors)}/{len(test_errors)}"
)

## E. Per-snapshot detail

Scatter plots relate graph size (active nodes, edges) to reconstruction error.

In [ ]:
train_set = {id(r) for r in train_recs}
test_set = {id(r) for r in test_recs}

rows = []
for r in gae_eligible:
    x, ei, _ = record_to_tensors(r)
    err = graph_reconstruction_error(model, x, ei)
    if id(r) in train_set:
        split = "train"
    elif id(r) in test_set:
        split = "test"
    else:
        split = "unused"
    rows.append(
        {
            "package": r.package,
            "session_id": r.session_id,
            "window": r.window_index,
            "split": split,
            "n_active_nodes": r.n_active_nodes,
            "n_edges": r.n_edges,
            "events_kept": r.events_kept,
            "reconstruction_error": err,
        }
    )

detail = pd.DataFrame(rows).sort_values("reconstruction_error")
display(detail.head(10))
display(detail.tail(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(
    data=detail, x="n_active_nodes", y="reconstruction_error", hue="split",
    alpha=0.7, ax=axes[0], palette={"train": "#4c72b0", "test": "#dd8452", "unused": "#999"}
)
axes[0].set_title("Active nodes vs error")
sns.scatterplot(
    data=detail, x="n_edges", y="reconstruction_error", hue="split",
    alpha=0.7, ax=axes[1], palette={"train": "#4c72b0", "test": "#dd8452", "unused": "#999"}
)
axes[1].set_title("Edges vs error")
plt.tight_layout()
plt.show()

## F. Inspect one graph (optional)

Pick a snapshot and visualize which of the 22 categories were active.

In [ ]:
# Change index to explore different snapshots
PICK = 0
rec = gae_eligible[PICK]
g = rec.graph
assert g is not None

active = {c: g.nodes[c].act_count for c in GRAPH_CATEGORY_UNIVERSE if g.nodes[c].act_count > 0}
act_df = pd.Series(active).sort_values(ascending=True)

mask = detail.session_id == rec.session_id
if rec.window_index is not None:
    mask &= detail.window == rec.window_index
err_val = float(detail.loc[mask, "reconstruction_error"].iloc[0])

fig, ax = plt.subplots(figsize=(8, max(4, len(act_df) * 0.25)))
act_df.plot(kind="barh", ax=ax, color="#4c72b0")
ax.set_xlabel("act_count")
ax.set_title(f"{rec.package} | w{rec.window_index} | {rec.n_edges} edges | error={err_val:.3f}")
plt.tight_layout()
plt.show()

print("Active categories:", list(active.keys()))
print("Edges (top 10 by w_cum):")
edge_rows = sorted(
    ((u, v, e.w_cum) for (u, v), e in g.edges.items()),
    key=lambda t: -t[2],
)[:10]
for u, v, w in edge_rows:
    print(f"  {u} → {v}  w_cum={w:.0f}")

## G. Save artifacts (optional)

In [ ]:
SAVE = True

if SAVE:
    model_path = OUTPUT_DIR / "gae_corpus_model.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "in_channels": node_feature_dim(),
            "hidden_channels": GAE_HIDDEN_DIM,
            "epochs": EPOCHS,
            "lr": GAE_LR,
            "window_mode": window_mode.value,
            "window_sec": WINDOW_SEC,
            "snapshots": SNAPSHOTS,
        },
        model_path,
    )
    detail.to_csv(OUTPUT_DIR / "per_snapshot_errors.csv", index=False)
    pd.DataFrame({"epoch": range(1, len(curve) + 1), "loss": curve}).to_csv(
        OUTPUT_DIR / "training_curve.csv", index=False
    )
    print(f"Saved model → {model_path}")
    print(f"Saved tables → {OUTPUT_DIR}")